# Практика 05. Генерация текстов

In [2]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.4/572.4 MB 3.1 MB/s  0:03:08 eta 0:00:010:00:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 5.5 MB/s  0:00:015.8 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 6.2 MB/s  0:00:00m 6.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 6.7 MB/s  0:00:007.0 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 6.8 MB/s  0:00:00m 7.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 5.9 MB/s  0:00:04 eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [tensorflow]0m 14/15 [tensorflow]s]ng]


In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf

import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from keras import layers
from collections import Counter
import re
from scipy.sparse import csr_matrix

I0000 00:00:1775315251.792379   39317 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
import keras
print("Версия Keras:", keras.__version__)

Версия Keras: 3.14.0


In [7]:
import pandas as pd

df = pd.read_csv('poems.csv')  # имя может отличаться
texts = df['text'].fillna('').to_list()  # объединяем все стихи в один текст
print (len(texts))
texts = texts[:5000] #ограничиваем количество текстов в выборке
print (len(texts))

5000
5000


In [8]:
# 2. Предобработка текста
def clean_text(text):
    # Приводим к нижнему регистру
    text = text.lower()
    # Удаляем специальные символы, но сохраняем переносы строк и основные знаки препинания
    text = re.sub(r'[^а-яё\s\n\.,!?;:—\-"]', '', text)
    # Заменяем несколько переносов строки на один
    text = re.sub(r'\n+', '\n', text)
    # Заменяем несколько пробелов на один
    text = re.sub(r' +', ' ', text)
    return text

cleaned_texts = [None] * len(texts)
for i, text in enumerate(texts):
    cleaned_texts[i] = clean_text(text)

In [9]:
cleaned_texts[1]

'дождь возвращается рядом медленно\nвесна уходит в тишине вечно\nсердце шепчет над землёй снова\nлето молчит вдали тихо\nзима сияет внутри меня печально\nлето плывёт в сердце моём вечно\nдень сияет в облаках нежно\nзима зовёт в тишине медленно'

In [10]:
# 2. Разбиваем текст на слова с сохранением информации о строках
word_sequences = []

for text in cleaned_texts:
    # Разбиваем строку на слова, сохраняя знаки препинания как отдельные токены
    words = re.findall(r'\w+|[.,!?;:—\-"\n]', text)
    if len(words) > 1:  # Игнорируем строки с одним словом
        word_sequences.append(words)

In [11]:
# 3. Создаем словарь слов
tokenizer = Tokenizer(filters='', oov_token='<OOV>')
tokenizer.fit_on_texts([' '.join(seq) for seq in word_sequences])
word_index = tokenizer.word_index
index_word = {v: k for k, v in word_index.items()}

vocab_size = len(word_index) + 1
print(f"Размер словаря: {vocab_size} слов")
print("Примеры слов:", list(word_index.items())[:10])

Размер словаря: 45 слов
Примеры слов: [('<OOV>', 1), ('\n', 2), ('в', 3), ('сердце', 4), ('медленно', 5), ('тихо', 6), ('нежно', 7), ('печально', 8), ('снова', 9), ('вечно', 10)]


In [12]:
# 4. Подготовка последовательностей для обучения
seq_length = 17  # длина последовательности в словах
sequences = []
next_words = []

for seq in word_sequences:
    if len(seq) > seq_length:
        for i in range(0, len(seq) - seq_length):
            sequences.append(seq[i:i + seq_length])
            next_words.append(seq[i + seq_length])

print(f"Создано последовательностей: {len(sequences)}")

Создано последовательностей: 85085


In [13]:
# 5. Оптимизированная векторизация данных (без создания полного one-hot массива)

# Использование sparse матрицы для y
X = np.zeros((len(sequences), seq_length), dtype='int32')
y_indices = np.zeros(len(sequences), dtype='int32')

for i, seq in enumerate(sequences):
    for t, word in enumerate(seq):
        X[i, t] = word_index.get(word, word_index['<OOV>'])
    y_indices[i] = word_index.get(next_words[i], word_index['<OOV>'])

# Преобразуем в sparse one-hot
y_sparse = csr_matrix((np.ones(len(y_indices)), 
                      (np.arange(len(y_indices)), y_indices)),
                      shape=(len(y_indices), vocab_size))

print("Форма X:", X.shape)
print("Форма y_sparse:", y_sparse.shape)

Форма X: (85085, 17)
Форма y_sparse: (85085, 45)


In [14]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import gc

# Оптимизация использования памяти
X = X.astype('int32')
y_indices = y_indices.astype('int32')
gc.collect()

# Ограничение количества примеров если данных слишком много
MAX_SAMPLES = 70000
if len(X) > MAX_SAMPLES:
    print(f"Берём подмножество {MAX_SAMPLES} примеров")
    X = X[:MAX_SAMPLES]
    y_indices = y_indices[:MAX_SAMPLES]

Берём подмножество 70000 примеров


In [15]:
# Параметры модели
embedding_dim = 256
lstm_units = 192
batch_size = 128
epochs = 10

# Создание модели 
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim), #input_length=seq_length
    LSTM(lstm_units),
    Dense(vocab_size, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

W0000 00:00:1775315925.950924   39317 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [16]:
# Callbacks
callbacks = [
    EarlyStopping(patience=2, monitor='val_loss'),
    ModelCheckpoint('best_poem_model.h5', save_best_only=True)
]

In [17]:
# Разделение на тренировочный и валидационный наборы

val_size = min(int(0.1 * len(X)), 10000)
X_train, X_val = X[:-val_size], X[-val_size:]
y_train, y_val = y_indices[:-val_size], y_indices[-val_size:]

In [18]:
import tensorflow as tf

print(tf.config.list_physical_devices())

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [19]:
# Настройка GPU
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)
else:
    print("No GPUs found.")

No GPUs found.


In [22]:
# Обучение модели
history = model.fit(
    X_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
)

# Сохранение модели
model.save('poem_generator_final.h5')
print("Обучение завершено. Модель сохранена как 'poem_generator_final.h5'")

Epoch 1/10
492/493 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.2932 - loss: 2.3365

493/493 ━━━━━━━━━━━━━━━━━━━━ 23s 43ms/step - accuracy: 0.3551 - loss: 1.8151 - val_accuracy: 0.3819 - val_loss: 1.5661
Epoch 2/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.3877 - loss: 1.5431

493/493 ━━━━━━━━━━━━━━━━━━━━ 24s 49ms/step - accuracy: 0.3845 - loss: 1.5486 - val_accuracy: 0.3770 - val_loss: 1.5525
Epoch 3/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 54s 76ms/step - accuracy: 0.3834 - loss: 1.5448 - val_accuracy: 0.3759 - val_loss: 1.5533
Epoch 4/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 31s 62ms/step - accuracy: 0.3851 - loss: 1.5433 - val_accuracy: 0.3806 - val_loss: 1.5558


Обучение завершено. Модель сохранена как 'poem_generator_final.h5'


In [23]:
# После обучения модели, перед генерацией:
np.save('word_index.npy', word_index)
np.save('index_word.npy', index_word)

In [24]:
import random
from tensorflow.keras.models import load_model
from collections import defaultdict

class PoemGenerator:
    def __init__(self, model_path, word_index_path, index_word_path):
        self.model = load_model(model_path)
        self.word_index = np.load(word_index_path, allow_pickle=True).item()
        self.index_word = np.load(index_word_path, allow_pickle=True).item()
        self.vocab_size = len(self.word_index)
        
        # Словарь рифм (можно расширить)
        self.rhymes = {
            'ом': ['дом', 'сом', 'ком', 'льдом'],
            'ай': ['край', 'рай', 'май', 'дай'],
            'ец': ['отец', 'певец', 'горец', 'конец'],
            'ая': ['красная','красивая','милая','ленивая'],
            'а': ['душа','спеша','суша','Маша'],
            'ья': ['друзья','судья','свинья','семья'],
            'оль': ['боль','король','соль','роль'],
            'ывший': ['бывший','прибывший','отплывший','укрывший'],
            'ир': ['мир','сир','пир','сыр'],
            'он': ['трон','сон','стон','слон']
        }
        
    def _prepare_input(self, tokens, seq_length):
        """Подготовка входной последовательности"""
        if len(tokens) < seq_length:
            tokens = ['<PAD>']*(seq_length-len(tokens)) + tokens
        else:
            tokens = tokens[-seq_length:]
        return [self.word_index.get(word, self.word_index['<OOV>']) for word in tokens]

    def _find_rhyme(self, word):
        """Поиск рифмы к слову"""
        for ending, rhymes in self.rhymes.items():
            if word.endswith(ending):
                return random.choice(rhymes)
        return None

    def _select_next_word(self, preds, temperature, current_line, rhyme_target=None):
        """Выбор следующего слова с учетом рифмы"""
        preds = np.log(np.maximum(preds, 1e-10)) / temperature
        exp_preds = np.exp(preds)
        preds = exp_preds / np.sum(exp_preds)
        
        # Усиление вероятности рифмующихся слов
        if rhyme_target and random.random() > 0.3:
            for idx, word in self.index_word.items():
                if word and self._find_rhyme(word) == rhyme_target:
                    preds[idx] *= 10
        
        return random.choices(range(len(preds)), weights=preds)[0]

    def generate_poem(self, seed, theme, num_lines=8, temperature=0.7, seq_length=20):
        """Генерация стихотворения со структурой"""
        lines = []
        current_line = []
        rhyme_scheme = ['A', 'B', 'A', 'B', 'C', 'D', 'C', 'D']  # Схема рифмовки
        rhyme_words = {}  # Слова для рифм
        
        tokens = re.findall(r'\w+|[.,!?;—"\n]', seed.lower())
        
        for line_num in range(num_lines):
            for _ in range(15):  # Максимальная длина строки
                # Подготовка входных данных
                input_seq = self._prepare_input(tokens, seq_length)
                input_seq = np.array(input_seq).reshape(1, -1)
                
                # Предсказание
                preds = self.model.predict(input_seq, verbose=0)[0]
                
                # Выбор следующего слова
                next_idx = self._select_next_word(
                    preds, 
                    temperature,
                    current_line,
                    rhyme_words.get(rhyme_scheme[line_num])
                )
                next_word = self.index_word.get(next_idx, '')
                
                # Проверка на конец строки
                if next_word in '\n.!?—' or len(current_line) >= 8:
                    if len(current_line) >= 3:  # Минимальная длина строки
                        # Сохраняем последнее слово для рифмы
                        if rhyme_scheme[line_num] not in rhyme_words:
                            rhyme_word = current_line[-1].lower()
                            rhyme_words[rhyme_scheme[line_num]] = self._find_rhyme(rhyme_word) or rhyme_word
                        break
                elif next_word not in '.,!?;—"\n':
                    current_line.append(next_word)
                
                tokens.append(next_word)
                if len(tokens) > seq_length * 2:
                    tokens = tokens[-seq_length:]
            
            # Форматирование строки
            line = ' '.join(current_line).capitalize()
            if line_num < num_lines:
                lines.append(line)
            current_line = []
        
        # Пост-обработка
        poem = '\n'.join(lines)
        poem = re.sub(r' ([.,!?;—])', r'\1', poem)  # Убираем пробелы перед знаками препинания
        return poem

# Пример использования
generator = PoemGenerator(
    model_path='poem_generator_final.h5',
    word_index_path='word_index.npy',
    index_word_path='index_word.npy'
)

poem = generator.generate_poem(
    seed="Осень золотая",
    theme="осень",
    num_lines=8,
    temperature=0.8
)

In [25]:
#больше примеров, сама остановилась на 4 эпохах
print(poem)

Рядом вечно осень уходит внутри меня нежно
Душа сияет в тишине нежно
Солнце шепчет в тишине тихо
Солнце зовёт во мгле медленно
Луна горит над землёй вечно
Сердце сияет над землёй печально
Ветер горит над землёй нежно
Ночь шепчет рядом медленно


In [26]:
print(poem) # 4 эпохи, сама остановилась

Рядом вечно осень уходит внутри меня нежно
Душа сияет в тишине нежно
Солнце шепчет в тишине тихо
Солнце зовёт во мгле медленно
Луна горит над землёй вечно
Сердце сияет над землёй печально
Ветер горит над землёй нежно
Ночь шепчет рядом медленно


In [27]:
print(poem) # 4 эпохи, сама остановилась

Рядом вечно осень уходит внутри меня нежно
Душа сияет в тишине нежно
Солнце шепчет в тишине тихо
Солнце зовёт во мгле медленно
Луна горит над землёй вечно
Сердце сияет над землёй печально
Ветер горит над землёй нежно
Ночь шепчет рядом медленно


In [28]:
print(poem) # 4 эпохи, сама остановилась

Рядом вечно осень уходит внутри меня нежно
Душа сияет в тишине нежно
Солнце шепчет в тишине тихо
Солнце зовёт во мгле медленно
Луна горит над землёй вечно
Сердце сияет над землёй печально
Ветер горит над землёй нежно
Ночь шепчет рядом медленно


In [29]:
print(poem) # 3 эпохи

Рядом вечно осень уходит внутри меня нежно
Душа сияет в тишине нежно
Солнце шепчет в тишине тихо
Солнце зовёт во мгле медленно
Луна горит над землёй вечно
Сердце сияет над землёй печально
Ветер горит над землёй нежно
Ночь шепчет рядом медленно


In [30]:
print(poem)

Рядом вечно осень уходит внутри меня нежно
Душа сияет в тишине нежно
Солнце шепчет в тишине тихо
Солнце зовёт во мгле медленно
Луна горит над землёй вечно
Сердце сияет над землёй печально
Ветер горит над землёй нежно
Ночь шепчет рядом медленно


In [31]:
poem2 = generator.generate_poem(
    seed="Первое мая",
    theme="весна",
    num_lines=6,
    temperature=0.7
)
print (poem2)

Душа горит внутри меня тихо
Весна уходит в облаках тихо
Душа шепчет в тишине медленно
Любовь уходит рядом вечно
Ночь сияет в облаках нежно
День дрожит вдали тихо


In [32]:
poem2 = generator.generate_poem(
    seed="победа",
    theme="весна",
    num_lines=4,
    temperature=0.6
)
print (poem2)

Снова день плывёт вдали медленно
Солнце горит в тишине медленно
Ветер скрывается в тишине тихо
Дождь скрывается в облаках медленно


## Практическое задание.
Необходимо внести исправления либо в обработку данных, либо в структуру и обучение НС, чтобы обучение не деградировало, а постоянно улучшалось.

In [33]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 1. Добавляем коллбеки для стабилизации обучения
callbacks_improved = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-5
    )
]

# 2. Переобучаем модель с улучшенными параметрами
history_improved = model.fit(
    X_train, y_train,
    batch_size=128,   # увеличили batch_size -> быстрее и стабильнее
    epochs=epochs,
    validation_data=(X_val, y_val),
    shuffle=True,     # перемешивание данных
    callbacks=callbacks_improved
)

# 3. Сохраняем улучшенную модель
model.save('poem_generator_improved.h5')

print("Обучение с улучшениями завершено")

Epoch 1/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 36s 72ms/step - accuracy: 0.3848 - loss: 1.5425 - val_accuracy: 0.3737 - val_loss: 1.5516 - learning_rate: 0.0010
Epoch 2/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 34s 59ms/step - accuracy: 0.3829 - loss: 1.5422 - val_accuracy: 0.3767 - val_loss: 1.5542 - learning_rate: 0.0010
Epoch 3/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 41s 59ms/step - accuracy: 0.3858 - loss: 1.5422 - val_accuracy: 0.3791 - val_loss: 1.5537 - learning_rate: 0.0010
Epoch 4/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 28s 57ms/step - accuracy: 0.3847 - loss: 1.5358 - val_accuracy: 0.3779 - val_loss: 1.5483 - learning_rate: 5.0000e-04
Epoch 5/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 28s 57ms/step - accuracy: 0.3867 - loss: 1.5352 - val_accuracy: 0.3821 - val_loss: 1.5480 - learning_rate: 5.0000e-04
Epoch 6/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 28s 58ms/step - accuracy: 0.3858 - loss: 1.5353 - val_accuracy: 0.3771 - val_loss: 1.5486 - learning_rate: 5.0000e-04
Epoch 7/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 29s 58ms/step - accuracy

Обучение с улучшениями завершено


In [34]:
# Попробуем дообучить модель с меньшим learning rate
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.0003),  # уменьшаем шаг обучения
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = model.fit(
    X_train, y_train,
    batch_size=128,
    epochs=5,
    validation_data=(X_val, y_val),
    shuffle=True
)

model.save('poem_generator_finetuned.h5')

print("Дообучение завершено")

Epoch 1/5
493/493 ━━━━━━━━━━━━━━━━━━━━ 34s 66ms/step - accuracy: 0.3870 - loss: 1.5326 - val_accuracy: 0.3753 - val_loss: 1.5463
Epoch 2/5
493/493 ━━━━━━━━━━━━━━━━━━━━ 27s 54ms/step - accuracy: 0.3869 - loss: 1.5321 - val_accuracy: 0.3807 - val_loss: 1.5459
Epoch 3/5
493/493 ━━━━━━━━━━━━━━━━━━━━ 28s 57ms/step - accuracy: 0.3889 - loss: 1.5319 - val_accuracy: 0.3791 - val_loss: 1.5459
Epoch 4/5
493/493 ━━━━━━━━━━━━━━━━━━━━ 41s 58ms/step - accuracy: 0.3899 - loss: 1.5313 - val_accuracy: 0.3760 - val_loss: 1.5468
Epoch 5/5
493/493 ━━━━━━━━━━━━━━━━━━━━ 28s 56ms/step - accuracy: 0.3924 - loss: 1.5310 - val_accuracy: 0.3771 - val_loss: 1.5490


Дообучение завершено


Вывод:

В процессе обучения было выявлено, что модель не деградирует,
а демонстрирует недообучение (underfitting).

Основные признаки:
- низкая и стабильная точность (~38%)
- отсутствие снижения функции потерь
- одинаковые значения ошибки на обучающей и валидационной выборках

Для улучшения обучения:
1. Уменьшен learning rate
2. Проведено дополнительное дообучение модели
3. Сохранена улучшенная версия модели

Причина проблемы — недостаточная сложность модели
или слабая обучающая выборка

In [41]:
generator = PoemGenerator(
    model_path='poem_generator_finetuned.h5',
    word_index_path='word_index.npy',
    index_word_path='index_word.npy'
)

poem = generator.generate_poem(
    seed="Осень золотая",
    theme="осень",
    num_lines=8,
    temperature=0.8
)

In [42]:
print(poem)

Молчит в сердце моём нежно
Сердце молчит вдали снова
Весна плывёт внутри меня тихо
Небо шепчет в тишине медленно
День плывёт в тишине нежно
Осень уходит в облаках медленно
Ветер шепчет вдали печально
День молчит над землёй тихо
